In [2]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import psycopg2
from sqlalchemy import create_engine
import warnings
from datetime import datetime, timedelta

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
warnings.filterwarnings('ignore')

# Configure display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


In [3]:
# Database connection parameters
# Note: Run the connection in the terminal first:
# psql -h 127.0.0.1 -p 5432 -U dfstechbi -d db_fraud
from urllib.parse import quote_plus
from sqlalchemy import text

DB_CONFIG = {
    'host': '127.0.0.1',
    'port': '5432',
    'database': 'db_fraud',
    'user': 'dfstechbi',
    'password': 'DfsTeChB1@923'  # You'll enter this when prompted
}

# Prompt for password
from getpass import getpass
DB_CONFIG['password'] = quote_plus(getpass('Enter database password: '))

# Create SQLAlchemy engine
connection_string = f"postgresql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
engine = create_engine(connection_string)

# Test connection
try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT version();"))
        print("✅ Database connection successful!")
        print(f"PostgreSQL version: {result.fetchone()[0].split(',')[0]}")
except Exception as e:
    print(f"❌ Connection failed: {e}")

✅ Database connection successful!
PostgreSQL version: PostgreSQL 17.6 on x86_64-pc-linux-gnu


In [18]:
# Load fraud data from July 6 to July 31
query = """
SELECT * 
FROM public.fraud 
WHERE DATE(transaction_datetime) BETWEEN '2025-07-06' AND '2025-07-31'
"""

# Execute query and load data into DataFrame
fraud_data = pd.read_sql_query(query, engine)

print(f"✅ Fraud data loaded successfully!")
print(f"📊 Total transactions: {len(fraud_data):,}")
print(f"📅 Date range: {fraud_data['transaction_datetime'].min()} to {fraud_data['transaction_datetime'].max()}")
print(f"🔍 Columns: {list(fraud_data.columns)}")
print(f"💾 Memory usage: {fraud_data.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Display first few rows
fraud_data.head()

✅ Fraud data loaded successfully!
📊 Total transactions: 4,994
📅 Date range: 2025-07-06 00:49:10 to 2025-07-31 21:58:05
🔍 Columns: ['complaint_num', 'trans_id', 'complaint_msisdn', 'fraud_msisdn', 'victim_msisdn', 'ac_from', 'ac_to', 'trx_amount', 'trx_channel', 'trx_type', 'created_datetime', 'resolved_datetime', 'transaction_datetime']
💾 Memory usage: 3.39 MB


,complaint_num,trans_id,complaint_msisdn,fraud_msisdn,victim_msisdn,ac_from,ac_to,trx_amount,trx_channel,trx_type,created_datetime,resolved_datetime,transaction_datetime
0,COM3138748,84718666265,y9alllKUxGGtp+EPVWRk5g==,9fM3d83phtikqnJZXCtcsw==,y9alllKUxGGtp+EPVWRk5g==,y9alllKUxGGtp+EPVWRk5g==,SXnZTu7qEm3ZFZ7TKtz7Eg==,5000,NEW_JC_APP,Transfer(C2C),2025-07-14 14:57:04,2025-07-14 15:27:28,2025-07-14 12:59:32
1,COM3146015,84790998973,5agDegbMX+Lke9YSASRUUg==,TwH/d1aylJxP/z2zcaSV6g==,5agDegbMX+Lke9YSASRUUg==,bUZJ4oFcMkS1dEuD2P+FtA==,5agDegbMX+Lke9YSASRUUg==,600,THIRD_PARTY_WEB,Get Loan,2025-07-15 13:50:39,2025-07-15 14:09:17,2025-07-15 12:40:57
2,COM3163397,84888781805,kjR08VP3zqyWalGyq9I5ww==,5mSkp7zPTZlVdGWIJihw1A==,kjR08VP3zqyWalGyq9I5ww==,kjR08VP3zqyWalGyq9I5ww==,2gLyj563iWyTJCN8HtM7nw==,2500,NEW_JC_APP,Transfer(C2C),2025-07-17 16:30:16,2025-07-17 16:47:41,2025-07-16 19:09:55
3,COM3182402,85067518861,3ARXLxJ5K0layg3LJkJKTg==,dCTq6yXfIL5KsLxVDyDkbQ==,3ARXLxJ5K0layg3LJkJKTg==,bUZJ4oFcMkS1dEuD2P+FtA==,3ARXLxJ5K0layg3LJkJKTg==,2000,THIRD_PARTY_WEB,Get Loan,2025-07-19 23:47:33,2025-07-19 23:52:43,2025-07-19 11:24:09
4,COM3194368,85230820654,SIbhyRY3k2NGVSKHO9fuFQ==,sMr0ftwRE93+i7uPRrpq4A==,SIbhyRY3k2NGVSKHO9fuFQ==,SIbhyRY3k2NGVSKHO9fuFQ==,vrCCZUVhfr0swb+ihcMAjQ==,7800,NEW_JC_APP,Transfer(C2C),2025-07-21 19:23:36,2025-07-21 19:26:49,2025-07-21 17:49:47


In [7]:
# Load fraud accounts with types from parquet file
fraud_accounts_with_types = pd.read_parquet('../data/fraud_accounts_with_types')

# Filter for Customer Account type
customer_accounts = fraud_accounts_with_types[fraud_accounts_with_types['account_type_name'] == 'Customer Account']

print(f"✅ Fraud accounts data loaded successfully!")
print(f"📊 Total accounts: {len(fraud_accounts_with_types):,}")
print(f"👤 Customer accounts: {len(customer_accounts):,}")
print(f"🔍 Columns: {list(fraud_accounts_with_types.columns)}")

# Display first few rows of customer accounts
customer_accounts.head()

✅ Fraud accounts data loaded successfully!
📊 Total accounts: 5,519
👤 Customer accounts: 5,276
🔍 Columns: ['a_c_reference', 'account_type_name']


,a_c_reference,account_type_name
0,A4IodXocoJOsy7gXHlR2yA==,Customer Account
1,eP+92RAjQ+bwD8u8I2w4FA==,Customer Account
2,/HbE5CTocvF9+aJxApc+aQ==,Customer Account
3,v3jU6QuWv8diZu3lc2I4GA==,Customer Account
5,VzfjB//rB1H/jwmEMApF0g==,Customer Account


In [ ]:
# Query stixor_iar table for customer accounts between 2025-06-01 and 2025-07-05
# Get fraud IDs that are in customer accounts
# fraud_customer_ids = fraud_data[fraud_data['ac_from'].isin(customer_accounts['a_c_reference'])]['ac_from'].unique()
# fraud_ids = tuple(fraud_customer_ids.tolist())

query = f"""
SELECT * 
FROM public.stixor_iar 
WHERE DATE(data_date) BETWEEN '2025-07-01' AND '2025-07-31'
"""

# Execute query and load data into DataFrame
fraud_txns = pd.read_sql_query(query, engine)

print(f"✅ Stixor IAR data loaded successfully!")
print(f"📊 Total transactions: {len(fraud_txns):,}")
if len(fraud_txns) > 0:
    print(f"📅 Date range: {fraud_txns['data_date'].min()} to {fraud_txns['data_date'].max()}")
    print(f"🔍 Columns: {list(fraud_txns.columns)}")
    print(f"💾 Memory usage: {fraud_txns.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Display first few rows
fraud_txns.head()

✅ Stixor IAR data loaded successfully!
📊 Total transactions: 733
📅 Date range: 2025-06-01 to 2025-07-05
🔍 Columns: ['data_date', 'trans_id', 'trans_initiate_time', 'customer_msisdn', 'trx_channel', 'trx_type', 'trx_status', 'ac_from', 'ac_to', 'start_balance', 'trx_amt', 'end_balance', 'utility_company', 'bill_ref_number', 'fee', 'fed', 'reason_type', 'pur_of_remit', 'ec', 'merchant_id']
💾 Memory usage: 0.61 MB


,data_date,trans_id,trans_initiate_time,customer_msisdn,trx_channel,trx_type,trx_status,ac_from,ac_to,start_balance,trx_amt,end_balance,utility_company,bill_ref_number,fee,fed,reason_type,pur_of_remit,ec,merchant_id
0,2025-06-11,82581465109,2025-06-11 00:51:43,7q8SmSGPkwcvEpB+nMxI4Q==,NEW_JC_APP,Transfer(C2C),Completed,13erXkP0WmfVbJscxkHAWw==,7q8SmSGPkwcvEpB+nMxI4Q==,46199.60,40000.00,86199.60,None,None,0.00,0.00,Customer Transfer to Customer via New JC APP,None,None,None
1,2025-06-10,82524222915,2025-06-10 04:28:26,FvkOp2NNablC/zz2dJm6wg==,QR Payment,Merchant Payment,Completed,13erXkP0WmfVbJscxkHAWw==,FvkOp2NNablC/zz2dJm6wg==,26261.23,2120.00,24141.23,None,QR-MerchID: 03004010614,21.20,2.92,Customer Performs Merchant Payment to New Merc...,None,None,None
2,2025-06-02,82016603675,2025-06-02 02:18:56,xIE/obtgxSJj3xXNQxqaiw==,NEW_JC_APP,Transfer(C2C),Completed,13erXkP0WmfVbJscxkHAWw==,xIE/obtgxSJj3xXNQxqaiw==,0.00,25000.00,25000.00,None,None,0.00,0.00,Customer Transfer to Customer via New JC APP,None,None,None
3,2025-06-13,82709714634,2025-06-13 06:09:00,Xdip6eqfKq2VZjP9I0by4g==,NEW_JC_APP,Transfer(C2C),Completed,13erXkP0WmfVbJscxkHAWw==,Xdip6eqfKq2VZjP9I0by4g==,1287.30,1200.00,2487.30,None,None,0.00,0.00,Customer Transfer to Customer via New JC APP,None,None,None
4,2025-06-04,82167145151,2025-06-04 08:55:42,+NhgJHCrumn3Nv+GZnipmg==,NEW_JC_APP,Transfer(C2C),Completed,13erXkP0WmfVbJscxkHAWw==,+NhgJHCrumn3Nv+GZnipmg==,576.75,1100.00,1676.75,None,None,0.00,0.00,Customer Transfer to Customer via New JC APP,None,None,None


In [10]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Define time windows in seconds
TIME_WINDOWS = {
    '1d': 86400,
    '5d': 432000,
    '10d': 864000,
    '20d': 1728000,
    '30d': 2592000
}

class UserFraudFeatureEngineer:
    """
    User-level feature engineering for fraud detection - creates one row per customer
    """
    
    def __init__(self, df):
        """
        Initialize with transaction dataframe
        Will aggregate all transactions per customer into user-level features
        """
        self.df = df.copy()
        
        # Convert timestamp to unix timestamp for window calculations
        self.df['unix_timestamp'] = pd.to_datetime(self.df['trans_initiate_time']).astype(int) // 10**9
        
        # Create a clean amount column (handle nulls)
        self.df['amount_clean'] = self.df['trx_amt'].fillna(0)
        
        # Store the reference date for relative calculations (use max date in dataset)
        self.reference_date = self.df['trans_initiate_time'].max()
        print(f"Using reference date for calculations: {self.reference_date}")
        
    def create_user_basic_features(self):
        """
        Create basic user-level aggregated features
        """
        print("Creating basic user-level features...")
        
        # Basic aggregations per user
        user_basic = self.df.groupby('ac_from').agg({
            # Transaction counts and timing
            'trans_id': 'count',
            'trans_initiate_time': ['min', 'max'],
            
            # Amount statistics
            'amount_clean': ['sum', 'mean', 'std', 'min', 'max', 'median'],
            
            # Balance statistics
            'start_balance': ['mean', 'min'],
            'end_balance': ['mean', 'max'],
            
            # Fee statistics
            'fee': ['sum', 'mean'],
            'fed': ['sum', 'mean'],
            
            # Channel and type diversity
            'trx_channel': 'nunique',
            'trx_type': 'nunique',
            'ac_to': 'nunique',
            'merchant_id': 'nunique',
            'utility_company': 'nunique'
        }).reset_index()
        
        # Flatten column names
        user_basic.columns = ['ac_from', 'total_transactions', 'first_transaction_date', 'last_transaction_date',
                             'total_amount', 'avg_amount', 'std_amount', 'min_amount', 'max_amount', 'median_amount',
                             'avg_start_balance', 'min_start_balance', 'avg_end_balance', 'max_end_balance',
                             'total_fees', 'avg_fee', 'total_fed', 'avg_fed',
                             'unique_channels', 'unique_transaction_types', 'unique_recipients', 'unique_merchants', 'unique_utilities']
        
        # Calculate additional percentiles manually
        percentile_features = self.df.groupby('ac_from')['amount_clean'].quantile([0.25, 0.75]).unstack()
        percentile_features.columns = ['q25_amount', 'q75_amount']
        percentile_features = percentile_features.reset_index()
        
        # Merge percentiles with main features
        user_basic = user_basic.merge(percentile_features, on='ac_from', how='left')
        
        # Calculate balance change
        balance_change = self.df.groupby('ac_from').apply(
            lambda x: (x['end_balance'] - x['start_balance']).mean()
        ).reset_index()
        balance_change.columns = ['ac_from', 'avg_balance_change']
        
        # Merge balance change
        user_basic = user_basic.merge(balance_change, on='ac_from', how='left')
        
        # Calculate account age in days
        user_basic['account_age_days'] = (user_basic['last_transaction_date'] - user_basic['first_transaction_date']).dt.days
        
        # Add derived features
        user_basic['transactions_per_day'] = np.where(
            user_basic['account_age_days'] > 0,
            user_basic['total_transactions'] / user_basic['account_age_days'],
            user_basic['total_transactions']
        )
        
        user_basic['amount_per_day'] = np.where(
            user_basic['account_age_days'] > 0,
            user_basic['total_amount'] / user_basic['account_age_days'],
            user_basic['total_amount']
        )
        
        user_basic['fee_to_amount_ratio'] = np.where(
            user_basic['total_amount'] > 0,
            user_basic['total_fees'] / user_basic['total_amount'],
            0
        )
        
        user_basic['channel_diversity_ratio'] = user_basic['unique_channels'] / user_basic['total_transactions']
        user_basic['recipient_diversity_ratio'] = user_basic['unique_recipients'] / user_basic['total_transactions']
        
        user_basic['amount_coefficient_variation'] = np.where(
            user_basic['avg_amount'] > 0,
            user_basic['std_amount'] / user_basic['avg_amount'],
            0
        )
        
        # Fill NaN values for fees and fed
        user_basic['total_fees'] = user_basic['total_fees'].fillna(0)
        user_basic['avg_fee'] = user_basic['avg_fee'].fillna(0)
        user_basic['total_fed'] = user_basic['total_fed'].fillna(0)
        user_basic['avg_fed'] = user_basic['avg_fed'].fillna(0)
        user_basic['unique_merchants'] = user_basic['unique_merchants'].fillna(0)
        user_basic['unique_utilities'] = user_basic['unique_utilities'].fillna(0)
        
        self.user_features = user_basic
        return self

In [11]:
# Create user-level features from stixor data
feature_engineer = UserFraudFeatureEngineer(stixor_data)
user_features = feature_engineer.create_user_basic_features().user_features

print(f"✅ User features created successfully!")
print(f"📊 Total users: {len(user_features):,}")
print(f"🔍 Feature columns: {len(user_features.columns)}")
print(f"💾 Memory usage: {user_features.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Display sample features
print("\n📋 Sample user features:")
user_features.head()

Using reference date for calculations: 2025-07-05 23:57:36
Creating basic user-level features...
✅ User features created successfully!
📊 Total users: 1,319
🔍 Feature columns: 33
💾 Memory usage: 0.42 MB

📋 Sample user features:


,ac_from,total_transactions,first_transaction_date,last_transaction_date,total_amount,avg_amount,std_amount,min_amount,max_amount,median_amount,avg_start_balance,min_start_balance,avg_end_balance,max_end_balance,total_fees,avg_fee,total_fed,avg_fed,unique_channels,unique_transaction_types,unique_recipients,unique_merchants,unique_utilities,q25_amount,q75_amount,avg_balance_change,account_age_days,transactions_per_day,amount_per_day,fee_to_amount_ratio,channel_diversity_ratio,recipient_diversity_ratio,amount_coefficient_variation
0,+AReE9QURBnUOZ5pWrh/wQ==,1,2025-06-02 03:26:21,2025-06-02 03:26:21,38.51,38.51,NaN,38.51,38.51,38.51,952541490.16,952541490.16,952541528.67,952541528.67,0.00,0.00,0.00,0.00,1,1,1,0,0,38.51,38.51,38.51,0,1.00,38.51,0.00,1.00,1.00,NaN
1,+E5AvQDMIgDz4OV1I7Va0A==,5,2025-07-01 10:44:55,2025-07-05 17:37:39,4200.00,840.00,0.00,840.00,840.00,840.00,NaN,NaN,NaN,NaN,0.00,0.00,0.00,0.00,1,1,1,0,0,840.00,840.00,NaN,4,1.25,1050.00,0.00,0.20,0.20,0.00
2,+I/eiTACtR9AILj7Yslt/Q==,7,2025-06-18 19:41:55,2025-07-05 08:26:37,31700.00,4528.57,3306.23,2000.00,9920.00,3000.00,7098.43,2008.52,4998.43,21017.91,72.00,10.29,9.93,1.42,2,3,3,0,0,2100.00,6290.00,-2100.00,16,0.44,1981.25,0.00,0.29,0.43,0.73
3,+LIGZt5SSaPSqbTKfRvtJQ==,1,2025-07-01 09:52:37,2025-07-01 09:52:37,5000.00,5000.00,NaN,5000.00,5000.00,5000.00,NaN,NaN,NaN,NaN,0.00,0.00,0.00,0.00,1,1,1,0,0,5000.00,5000.00,NaN,0,1.00,5000.00,0.00,1.00,1.00,NaN
4,+MGfuzzpmvkejnC1kLJJzw==,16,2025-06-04 03:08:41,2025-06-30 14:21:06,165410.00,10338.12,14103.56,60.00,49700.00,4975.00,9160.57,0.00,8998.69,39603.03,497.00,31.06,68.55,4.28,2,3,11,0,0,275.00,16000.00,-161.88,26,0.62,6361.92,0.00,0.12,0.69,1.36


In [14]:
fraud_txns_df = user_features.copy()

In [13]:
query = f"""
SELECT * 
FROM public.customer_fraud_basic_features
"""

# Execute query and load data into DataFrame
non_fraud_txns_df = pd.read_sql_query(query, engine)
non_fraud_txns_df.head()

,ac_from,total_transactions,first_transaction_date,last_transaction_date,account_age_days,total_amount,avg_amount,std_amount,min_amount,max_amount,median_amount,q25_amount,q75_amount,avg_start_balance,avg_end_balance,min_start_balance,max_end_balance,avg_balance_change,total_fees,avg_fee,total_fed,avg_fed,unique_channels,unique_transaction_types,unique_recipients,unique_merchants,unique_utilities,transactions_per_day,amount_per_day,fee_to_amount_ratio,channel_diversity_ratio,recipient_diversity_ratio,amount_coefficient_variation,batch_number,processing_timestamp,created_at
0,+/imZ8USd9KFuC4gZJ1GyQ==,4,2025-06-05 19:22:01+00:00,2025-06-30 06:23:42+00:00,24,23000.00,5750.00,6306.87,200.00,14500.00,2300.00,200.00,6000.00,6081.51,430.35,0.25,1455.50,-5651.16,4.02,1.00,0.63,0.16,1,2,2,0,1,0.17,958.33,0.00,0.25,0.50,1.10,1,2025-10-23 06:30:47.028873+00:00,2025-10-23 06:30:47.028873+00:00
1,+1bHkx/uqC8FpuRVKX1qPg==,6,2025-06-06 12:54:40+00:00,2025-07-03 16:55:30+00:00,27,21383.00,3563.83,4076.48,420.00,11200.00,1400.00,1170.00,5093.00,5813.97,2950.14,420.53,10493.18,-2863.83,0.00,0.00,0.00,0.00,1,3,2,0,1,0.22,791.96,0.00,0.17,0.33,1.14,1,2025-10-23 06:30:47.028873+00:00,2025-10-23 06:30:47.028873+00:00
2,+3RHK7kljTd3vBc9dxMuUw==,27,2025-06-03 06:40:09+00:00,2025-07-05 14:32:18+00:00,32,81295.00,3010.93,3540.20,100.00,15000.00,2000.00,700.00,4500.00,5395.04,7161.83,13.22,32058.15,1766.79,1.34,0.05,0.21,0.01,3,5,13,0,3,0.84,2540.47,0.00,0.11,0.48,1.18,1,2025-10-23 06:30:47.028873+00:00,2025-10-23 06:30:47.028873+00:00
3,+3tuRlhDLWIuIJSBVfpurA==,3,2025-06-02 09:25:15+00:00,2025-07-02 08:32:43+00:00,30,795.00,265.00,370.19,6.00,689.00,100.00,6.00,689.00,3.48,344.98,0.08,689.08,341.50,0.86,0.29,0.14,0.05,3,3,3,0,0,0.10,26.50,0.00,1.00,1.00,1.40,1,2025-10-23 06:30:47.028873+00:00,2025-10-23 06:30:47.028873+00:00
4,+4j9DDE/a4nfNKCA6Ghntw==,21,2025-06-01 08:56:16+00:00,2025-07-01 09:36:19+00:00,30,97100.00,4623.81,6307.34,60.00,25000.00,2500.00,500.00,5000.00,8918.12,8389.12,64.16,30252.61,-529.00,75.95,3.62,10.63,0.51,4,5,10,0,2,0.70,3236.67,0.00,0.19,0.48,1.36,1,2025-10-23 06:30:47.028873+00:00,2025-10-23 06:30:47.028873+00:00


In [15]:
len(non_fraud_txns_df)

281969

In [16]:
fraud_txns_df['fraud'] = 1

In [17]:
non_fraud_txns_df['fraud']=0

# Transaction Level Features

In [ ]:
query = f"""
SELECT * 
FROM public.stixor_iar 
WHERE DATE(data_date) BETWEEN '2025-07-01' AND '2025-07-31'
"""

# Execute query and load data into DataFrame
fraud_txns = pd.read_sql_query(query, engine)

In [ ]:
# Transaction-Level Fraud Feature Engineering Class
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

class TransactionFraudFeatureEngineer:
    """
    Transaction-level feature engineering for fraud detection
    Creates features for each individual transaction
    """
    
    def __init__(self, df, fraud_labels=None):
        """
        Initialize with transaction dataframe and fraud labels
        """
        self.df = df.copy()
        self.fraud_labels = fraud_labels
        
        # Convert timestamp to datetime
        self.df['trans_initiate_time'] = pd.to_datetime(self.df['trans_initiate_time'])
        self.df['unix_timestamp'] = self.df['trans_initiate_time'].astype(int) // 10**9
        
        # Clean amount column
        self.df['amount_clean'] = self.df['trx_amt'].fillna(0)
        
    def create_transaction_features(self):
        """
        Create transaction-specific features for each transaction
        """
        print("Creating transaction-level features...")
        
        # Basic transaction features
        self.df['hour_of_day'] = self.df['trans_initiate_time'].dt.hour
        self.df['day_of_week'] = self.df['trans_initiate_time'].dt.dayofweek
        self.df['is_weekend'] = self.df['day_of_week'].isin([5, 6]).astype(int)
        self.df['is_night'] = ((self.df['hour_of_day'] >= 22) | (self.df['hour_of_day'] <= 6)).astype(int)
        
        # Amount features
        self.df['amount_log'] = np.log1p(self.df['amount_clean'])
        self.df['fee_to_amount_ratio'] = np.where(
            self.df['amount_clean'] > 0,
            self.df['fee'].fillna(0) / self.df['amount_clean'],
            0
        )
        
        # Balance features
        self.df['balance_change'] = self.df['end_balance'] - self.df['start_balance']
        self.df['balance_utilization'] = np.where(
            self.df['start_balance'] > 0,
            self.df['amount_clean'] / self.df['start_balance'],
            0
        )
        
        return self
    
    def create_velocity_features(self, windows=[1, 5, 10, 30]):
        """
        Create velocity features based on historical transactions
        """
        print("Creating velocity features...")
        
        # Sort by user and timestamp
        self.df = self.df.sort_values(['ac_from', 'trans_initiate_time'])
        
        for window in windows:
            window_seconds = window * 24 * 3600  # Convert days to seconds
            
            # Count transactions in window
            self.df[f'txn_count_{window}d'] = self.df.groupby('ac_from')['unix_timestamp'].transform(
                lambda x: self._count_in_window(x, window_seconds)
            )
            
            # Sum amounts in window
            self.df[f'amount_sum_{window}d'] = self.df.groupby('ac_from').apply(
                lambda group: self._sum_in_window(group, 'amount_clean', window_seconds)
            ).reset_index(level=0, drop=True)
            
            # Average amount in window
            self.df[f'amount_avg_{window}d'] = np.where(
                self.df[f'txn_count_{window}d'] > 0,
                self.df[f'amount_sum_{window}d'] / self.df[f'txn_count_{window}d'],
                0
            )
        
        return self
    
    def create_behavioral_features(self):
        """
        Create behavioral pattern features
        """
        print("Creating behavioral features...")
        
        # Channel and type encoding
        le_channel = LabelEncoder()
        le_type = LabelEncoder()
        
        self.df['channel_encoded'] = le_channel.fit_transform(self.df['trx_channel'].fillna('unknown'))
        self.df['type_encoded'] = le_type.fit_transform(self.df['trx_type'].fillna('unknown'))
        
        # Recipient frequency (how often user sends to this recipient)
        recipient_freq = self.df.groupby(['ac_from', 'ac_to']).size().reset_index(name='recipient_frequency')
        self.df = self.df.merge(recipient_freq, on=['ac_from', 'ac_to'], how='left')
        
        # New recipient flag
        self.df['is_new_recipient'] = (self.df['recipient_frequency'] == 1).astype(int)
        
        # Time since last transaction
        self.df['time_since_last_txn'] = self.df.groupby('ac_from')['unix_timestamp'].diff().fillna(0)
        
        return self
    
    def create_contextual_features(self):
        """
        Create contextual features based on patterns
        """
        print("Creating contextual features...")
        
        # User's typical transaction amount percentiles
        user_amount_stats = self.df.groupby('ac_from')['amount_clean'].agg([
            'mean', 'std', 'median',
            lambda x: x.quantile(0.25),
            lambda x: x.quantile(0.75),
            lambda x: x.quantile(0.95)
        ]).reset_index()
        
        user_amount_stats.columns = ['ac_from', 'user_avg_amount', 'user_std_amount', 
                                    'user_median_amount', 'user_q25_amount', 
                                    'user_q75_amount', 'user_q95_amount']
        
        self.df = self.df.merge(user_amount_stats, on='ac_from', how='left')
        
        # Amount deviation from user norm
        self.df['amount_zscore'] = np.where(
            self.df['user_std_amount'] > 0,
            (self.df['amount_clean'] - self.df['user_avg_amount']) / self.df['user_std_amount'],
            0
        )
        
        # Amount percentile for user
        self.df['amount_above_q95'] = (self.df['amount_clean'] > self.df['user_q95_amount']).astype(int)
        self.df['amount_above_q75'] = (self.df['amount_clean'] > self.df['user_q75_amount']).astype(int)
        
        return self
    
    def prepare_modeling_data(self, fraud_column='fraud'):
        """
        Prepare final dataset for modeling
        """
        print("Preparing modeling data...")
        
        # Select feature columns
        feature_cols = [
            'amount_clean', 'amount_log', 'fee_to_amount_ratio', 'balance_change',
            'balance_utilization', 'hour_of_day', 'day_of_week', 'is_weekend', 'is_night',
            'channel_encoded', 'type_encoded', 'recipient_frequency', 'is_new_recipient',
            'time_since_last_txn', 'amount_zscore', 'amount_above_q95', 'amount_above_q75'
        ]
        
        # Add velocity features
        velocity_features = [col for col in self.df.columns if any(
            pattern in col for pattern in ['txn_count_', 'amount_sum_', 'amount_avg_']
        )]
        feature_cols.extend(velocity_features)
        
        # Create final dataset
        self.modeling_data = self.df[['trans_id', 'ac_from', 'trans_initiate_time'] + feature_cols].copy()
        
        # Add fraud labels if provided
        if self.fraud_labels is not None:
            self.modeling_data = self.modeling_data.merge(
                self.fraud_labels[['trans_id', fraud_column]], 
                on='trans_id', 
                how='left'
            )
            self.modeling_data[fraud_column] = self.modeling_data[fraud_column].fillna(0)
        
        return self
    
    def create_train_test_split(self, test_size=0.2, temporal_split=True, fraud_column='fraud'):
        """
        Create train/test splits with temporal considerations
        """
        print("Creating train/test splits...")
        
        if temporal_split:
            # Sort by time and split temporally
            self.modeling_data = self.modeling_data.sort_values('trans_initiate_time')
            split_point = int(len(self.modeling_data) * (1 - test_size))
            
            train_data = self.modeling_data.iloc[:split_point]
            test_data = self.modeling_data.iloc[split_point:]
            
            print(f"Temporal split - Train: {len(train_data):,}, Test: {len(test_data):,}")
            print(f"Train period: {train_data['trans_initiate_time'].min()} to {train_data['trans_initiate_time'].max()}")
            print(f"Test period: {test_data['trans_initiate_time'].min()} to {test_data['trans_initiate_time'].max()}")
        
        else:
            # Random split
            train_data, test_data = train_test_split(
                self.modeling_data, 
                test_size=test_size, 
                random_state=42, 
                stratify=self.modeling_data[fraud_column] if fraud_column in self.modeling_data.columns else None
            )
        
        # Separate features and labels
        feature_cols = [col for col in self.modeling_data.columns 
                       if col not in ['trans_id', 'ac_from', 'trans_initiate_time', fraud_column]]
        
        X_train = train_data[feature_cols]
        X_test = test_data[feature_cols]
        
        if fraud_column in self.modeling_data.columns:
            y_train = train_data[fraud_column]
            y_test = test_data[fraud_column]
            
            print(f"Fraud rate in train: {y_train.mean():.4f}")
            print(f"Fraud rate in test: {y_test.mean():.4f}")
            
            return X_train, X_test, y_train, y_test
        else:
            return X_train, X_test
    
    def _count_in_window(self, timestamps, window_seconds):
        """Helper function to count transactions in time window"""
        result = []
        for i, ts in enumerate(timestamps):
            window_start = ts - window_seconds
            count = sum(1 for t in timestamps[:i+1] if t >= window_start)
            result.append(count)
        return pd.Series(result, index=timestamps.index)
    
    def _sum_in_window(self, group, amount_col, window_seconds):
        """Helper function to sum amounts in time window"""
        result = []
        timestamps = group['unix_timestamp'].values
        amounts = group[amount_col].values
        
        for i, ts in enumerate(timestamps):
            window_start = ts - window_seconds
            amount_sum = sum(amounts[j] for j in range(i+1) if timestamps[j] >= window_start)
            result.append(amount_sum)
        
        return pd.Series(result, index=group.index)

# Usage example for transaction-level fraud detection
def create_transaction_fraud_dataset(transaction_data, fraud_labels):
    """
    Create a complete transaction-level fraud detection dataset
    """
    print("🚀 Creating transaction-level fraud detection dataset...")
    
    # Initialize feature engineer
    feature_engineer = TransactionFraudFeatureEngineer(transaction_data, fraud_labels)
    
    # Create all features
    feature_engineer.create_transaction_features()
    feature_engineer.create_velocity_features(windows=[1, 3, 7, 14, 30])
    feature_engineer.create_behavioral_features()
    feature_engineer.create_contextual_features()
    feature_engineer.prepare_modeling_data()
    
    # Create train/test splits
    X_train, X_test, y_train, y_test = feature_engineer.create_train_test_split(
        test_size=0.2, 
        temporal_split=True
    )
    
    print(f"✅ Dataset creation complete!")
    print(f"📊 Features: {X_train.shape[1]}")
    print(f"🎯 Train samples: {len(X_train):,} (fraud: {y_train.sum():,})")
    print(f"🎯 Test samples: {len(X_test):,} (fraud: {y_test.sum():,})")
    
    return X_train, X_test, y_train, y_test, feature_engineer

# Test file creation